# SegFormer fine-tuning on Google Colab (T4)

This  will:
- install all required libraries
- clone the full GitHub repository
- create an **empty mask** for every image without a labeled mask. Only a subset of images have masks. images without a mask are treated as **negative examples** with an all-zero mask
- fine-tune **SegFormer MIT-B2** by default (switch later to **MIT-B2**)
- evaluate the model and export sample predictions


In [ ]:
#@title 1. Install dependencies
!pip -q install -U transformers datasets evaluate accelerate huggingface_hub "pillow<12.0" matplotlib scikit-learn

In [ ]:
#@title 2. Imports and environment checks
import json
import os
import random
import string
from pathlib import Path
from google.colab import files

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont
from sklearn.model_selection import train_test_split
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. In Colab, go to Runtime -> Change runtime type -> T4 GPU.')


In [ ]:
#@title 3. Configuration
CHECKPOINT = 'nvidia/mit-b2'  # change to 'nvidia/mit-b2' later if wanted
OUTPUT_DIR = '/content/segformer-survey-maps-results'
REPO_URL = 'https://github.com/rijpma/survey-maps.git'
REPO_DIR = Path('/content/survey-maps')
DATA_ROOT = REPO_DIR / 'labelled' / 'batch1'
IMAGES_DIR = DATA_ROOT / 'images'
MASKS_DIR = DATA_ROOT / 'masks'
GENERATED_MASKS_DIR = DATA_ROOT / 'generated_masks'

IMAGE_SIZE = 512 #  switch to 384 for b2?
TEST_SIZE = 0.2
SEED = 42

NUM_EPOCHS = 20
TRAIN_BATCH_SIZE = 8 # switch to 4 for b2?
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 6e-5
WEIGHT_DECAY = 0.01
GRADIENT_ACCUMULATION_STEPS = 1

USE_AUGMENTATIONS = False
SKIP_ALL_AUGMENTATIONS = False
BW_PROBABILITY = 0.08
COLOR_PROBABILITY = 0.15
TEXT_PROBABILITY = 0.08
GRID_LINE_PROBABILITY = 0.08
BLUR_PROBABILITY = 0.06
NOISE_PROBABILITY = 0.06
TEXT_MIN_FONT_SIZE = 24
TEXT_MAX_FONT_SIZE = 36
TEXT_MAX_TEXTS = 1
TEXT_MIN_LENGTH = 3
TEXT_MAX_LENGTH = 12

id2label = {0: 'background', 1: 'object'}
label2id = {v: k for k, v in id2label.items()}

GENERATED_MASKS_DIR.mkdir(parents=True, exist_ok=True)

print('Checkpoint:', CHECKPOINT)
print('Output dir:', OUTPUT_DIR)
print('Repo URL:', REPO_URL)
print('Repo dir:', REPO_DIR)
print('Data root:', DATA_ROOT)
print('Image size:', IMAGE_SIZE)
print('Augmentations enabled:', USE_AUGMENTATIONS)
print('Skip all augmentations:', SKIP_ALL_AUGMENTATIONS)
print('BW probability:', BW_PROBABILITY)
print('Color probability:', COLOR_PROBABILITY)
print('Text probability:', TEXT_PROBABILITY)
print('Grid-line probability:', GRID_LINE_PROBABILITY)
print('Blur probability:', BLUR_PROBABILITY)
print('Noise probability:', NOISE_PROBABILITY)


In [ ]:
#@title 4. Clone repository
import subprocess

subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

print('Cloned repo to:', REPO_DIR)
print('Images found:', len(list(IMAGES_DIR.glob('*.png'))))
print('Masks found:', len(list(MASKS_DIR.glob('*.png'))))


In [ ]:
#@title 5. Build dataset records and generate empty masks where missing
def normalize_mask(mask_img):
    mask_arr = np.array(mask_img)
    if mask_arr.ndim == 3:
        mask_arr = mask_arr[..., 0]
    mask_arr = (mask_arr > 0).astype(np.uint8)
    return Image.fromarray(mask_arr, mode='L')

def make_empty_mask_like(image_path):
    img = Image.open(image_path)
    width, height = img.size
    return Image.fromarray(np.zeros((height, width), dtype=np.uint8), mode='L')

records = []
positive_count = 0
negative_count = 0

for image_path in sorted(IMAGES_DIR.glob('*.png')):
    mask_path = MASKS_DIR / image_path.name

    if mask_path.exists():
        normalized_mask = normalize_mask(Image.open(mask_path))
        generated_mask_path = GENERATED_MASKS_DIR / image_path.name
        normalized_mask.save(generated_mask_path)
        final_mask_path = generated_mask_path
        positive_count += 1
    else:
        empty_mask = make_empty_mask_like(image_path)
        generated_mask_path = GENERATED_MASKS_DIR / image_path.name
        empty_mask.save(generated_mask_path)
        final_mask_path = generated_mask_path
        negative_count += 1

    records.append({
        'image_path': str(image_path),
        'mask_path': str(final_mask_path),
        'has_object_mask': 1 if mask_path.exists() else 0,
        'file_name': image_path.name,
    })

print('Total records:', len(records))
print('Positive images with masks:', positive_count)
print('Negative images without masks:', negative_count)
assert len(records) > 0


In [ ]:
#@title 6. Stratified train/test split
indices = list(range(len(records)))
stratify_labels = [r['has_object_mask'] for r in records]

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=stratify_labels,
)

train_records = [records[i] for i in train_idx]
test_records = [records[i] for i in test_idx]

print('Train size:', len(train_records))
print('Test size:', len(test_records))
print('Train positives:', sum(r['has_object_mask'] for r in train_records))
print('Test positives:', sum(r['has_object_mask'] for r in test_records))


In [ ]:
#@title 7. Create Hugging Face datasets
train_ds = Dataset.from_list(train_records)
test_ds = Dataset.from_list(test_records)
raw_datasets = DatasetDict({'train': train_ds, 'test': test_ds})
raw_datasets


In [ ]:
#@title 8. Visual sanity check
def show_samples(dataset, n=4):
    n = min(n, len(dataset))
    fig, axes = plt.subplots(n, 3, figsize=(10, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row in range(n):
        example = dataset[row]
        image = Image.open(example['image_path']).convert('RGB')
        mask = Image.open(example['mask_path'])
        mask_np = np.array(mask)

        axes[row, 0].imshow(image)
        axes[row, 0].set_title(example['file_name'])
        axes[row, 0].axis('off')

        axes[row, 1].imshow(mask_np, cmap='gray', vmin=0, vmax=1)
        axes[row, 1].set_title(f"Mask (has_object={example['has_object_mask']})")
        axes[row, 1].axis('off')

        overlay = np.array(image).copy()
        overlay_mask = mask_np > 0
        overlay[overlay_mask] = [255, 0, 0]
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title('Overlay')
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()

show_samples(raw_datasets['train'], n=4)


In [ ]:
#@title 9. Processor and transforms
processor = SegformerImageProcessor.from_pretrained(
    CHECKPOINT,
    do_resize=True,
    size={'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    do_reduce_labels=False,
)

def load_image_and_mask(example):
    image = Image.open(example['image_path']).convert('RGB')
    mask = Image.open(example['mask_path'])
    mask_arr = np.array(mask)
    if mask_arr.ndim == 3:
        mask_arr = mask_arr[..., 0]
    mask_arr = (mask_arr > 0).astype(np.uint8)
    return image, mask_arr

def convert_to_bw_rgb(image, probability=0.2):
    if random.random() >= probability:
        return image
    return image.convert('L').convert('RGB')

def adjust_color_balance(
    image,
    probability=0.4,
    brightness_range=(0.85, 1.15),
    contrast_range=(0.85, 1.2),
    color_range=(0.75, 1.25),
    sharpness_range=(0.9, 1.1),
):
    if random.random() >= probability:
        return image

    out = image
    out = ImageEnhance.Brightness(out).enhance(random.uniform(*brightness_range))
    out = ImageEnhance.Contrast(out).enhance(random.uniform(*contrast_range))
    out = ImageEnhance.Color(out).enhance(random.uniform(*color_range))
    out = ImageEnhance.Sharpness(out).enhance(random.uniform(*sharpness_range))
    return out

def _random_place_name_string(min_length=3, max_length=12):
    if min_length > max_length:
        raise ValueError('min_length cannot be greater than max_length')

    lengths = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    weights = [18, 18, 16, 14, 10, 8, 6, 4, 3, 3]
    valid_pairs = [
        (length, weight)
        for length, weight in zip(lengths, weights)
        if min_length <= length <= max_length
    ]
    chosen_length = random.choices(
        [length for length, _ in valid_pairs],
        weights=[weight for _, weight in valid_pairs],
        k=1,
    )[0]

    all_digits = random.random() < 0.2
    if all_digits:
        alphabet = string.digits
    else:
        alphabet = string.ascii_uppercase

    return ''.join(random.choice(alphabet) for _ in range(chosen_length)).title()

def _load_text_font(font_size):
    font_candidates = [
        'DejaVuSans-Bold.ttf',
        'DejaVuSans.ttf',
    ]

    for font_path in font_candidates:
        try:
            return ImageFont.truetype(font_path, font_size)
        except OSError:
            continue

    return ImageFont.load_default()

def overlay_short_text(
    image,
    probability=0.15,
    mask=None,
    min_font_size=24,
    max_font_size=36,
    max_texts=1,
    alpha_range=(110, 190),
    text_color_choices=((0, 0, 0), (35, 35, 35), (70, 70, 70)),
    min_length=3,
    max_length=12,
):
    if random.random() >= probability:
        return image

    base = image.convert('RGBA')
    width, height = base.size
    n_texts = random.randint(1, max_texts)

    mask_array = None
    if mask is not None:
        if isinstance(mask, Image.Image):
            mask_array = np.array(mask)
        else:
            mask_array = np.asarray(mask)
        if mask_array.ndim == 3:
            mask_array = mask_array[..., 0]
        mask_array = (mask_array > 0).astype(np.uint8)

    for _ in range(n_texts):
        text = _random_place_name_string(min_length=min_length, max_length=max_length)
        font_size = random.randint(min_font_size, max_font_size)
        alpha = random.randint(*alpha_range)
        rgb = random.choice(text_color_choices)
        ink = (*rgb, alpha)

        font = _load_text_font(font_size)

        dummy = Image.new('RGBA', (1, 1), (0, 0, 0, 0))
        dummy_draw = ImageDraw.Draw(dummy)
        bbox = dummy_draw.textbbox((0, 0), text, font=font)
        text_w = max(1, bbox[2] - bbox[0])
        text_h = max(1, bbox[3] - bbox[1])

        pad_x = 8
        pad_y = 6
        text_layer = Image.new(
            'RGBA',
            (text_w + 2 * pad_x, text_h + 2 * pad_y),
            (0, 0, 0, 0),
        )
        text_draw = ImageDraw.Draw(text_layer)
        text_draw.text((pad_x, pad_y), text, fill=ink, font=font)

        max_x = max(0, width - text_layer.size[0])
        max_y = max(0, height - text_layer.size[1])

        x = random.randint(0, max_x) if max_x > 0 else 0
        y = random.randint(0, max_y) if max_y > 0 else 0

        if mask_array is not None and mask_array.any():
            ys, xs = np.nonzero(mask_array)
            anchor_index = random.randrange(len(xs))
            anchor_x = int(xs[anchor_index])
            anchor_y = int(ys[anchor_index])

            target_x = anchor_x - text_layer.size[0] // 2
            target_y = anchor_y - text_layer.size[1] // 2
            x = min(max(target_x, 0), max_x)
            y = min(max(target_y, 0), max_y)

        overlay = Image.new('RGBA', base.size, (0, 0, 0, 0))
        overlay.alpha_composite(text_layer, dest=(x, y))
        base = Image.alpha_composite(base, overlay)

    return base.convert('RGB')

def add_single_grid_line(
    image,
    probability=0.2,
    thickness_range=(2, 4),
    alpha_range=(120, 220),
    blur_radius_range=(0.0, 0.6),
):
    if random.random() >= probability:
        return image

    base = image.convert('RGBA')
    width, height = base.size
    overlay = Image.new('RGBA', base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    thickness = random.randint(*thickness_range)
    alpha = random.randint(*alpha_range)
    tone = random.choice([0, 20, 40, 60])
    color = (tone, tone, tone, alpha)

    if random.random() < 0.5:
        x = random.randint(0, max(0, width - 1))
        draw.line((x, 0, x, height), fill=color, width=thickness)
    else:
        y = random.randint(0, max(0, height - 1))
        draw.line((0, y, width, y), fill=color, width=thickness)

    blur_radius = random.uniform(*blur_radius_range)
    if blur_radius > 0:
        overlay = overlay.filter(ImageFilter.GaussianBlur(radius=blur_radius))

    return Image.alpha_composite(base, overlay).convert('RGB')

def apply_gaussian_blur(image, probability=0.15, radius_range=(0.3, 1.2)):
    if random.random() >= probability:
        return image

    radius = random.uniform(*radius_range)
    return image.filter(ImageFilter.GaussianBlur(radius=radius))

def add_speckle_noise(image, probability=0.15, std_range=(4.0, 14.0)):
    if random.random() >= probability:
        return image

    arr = np.asarray(image).astype(np.float32)
    std = random.uniform(*std_range)
    noise = np.random.normal(loc=0.0, scale=std, size=arr.shape)
    noisy = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)

def apply_map_augmentations(
    image,
    *,
    mask=None,
    skip_all=False,
    bw_probability=0.2,
    color_probability=0.4,
    text_probability=0.15,
    grid_line_probability=0.2,
    blur_probability=0.15,
    noise_probability=0.15,
    text_min_font_size=24,
    text_max_font_size=36,
    text_max_texts=1,
    text_min_length=3,
    text_max_length=12,
):
    out = image.convert('RGB')
    if skip_all:
        return out
    out = convert_to_bw_rgb(out, probability=bw_probability)
    out = adjust_color_balance(out, probability=color_probability)
    out = overlay_short_text(
        out,
        probability=text_probability,
        mask=mask,
        min_font_size=text_min_font_size,
        max_font_size=text_max_font_size,
        max_texts=text_max_texts,
        min_length=text_min_length,
        max_length=text_max_length,
    )
    out = add_single_grid_line(out, probability=grid_line_probability)
    out = apply_gaussian_blur(out, probability=blur_probability)
    out = add_speckle_noise(out, probability=noise_probability)
    return out

def train_transforms(example_batch):
    images = []
    labels = []
    for image_path, mask_path in zip(example_batch['image_path'], example_batch['mask_path']):
        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path)
        mask_arr = np.array(mask)
        if mask_arr.ndim == 3:
            mask_arr = mask_arr[..., 0]
        mask_arr = (mask_arr > 0).astype(np.uint8)

        if USE_AUGMENTATIONS:
            image = apply_map_augmentations(
                image,
                mask=mask_arr,
                skip_all=SKIP_ALL_AUGMENTATIONS,
                bw_probability=BW_PROBABILITY,
                color_probability=COLOR_PROBABILITY,
                text_probability=TEXT_PROBABILITY,
                grid_line_probability=GRID_LINE_PROBABILITY,
                blur_probability=BLUR_PROBABILITY,
                noise_probability=NOISE_PROBABILITY,
                text_min_font_size=TEXT_MIN_FONT_SIZE,
                text_max_font_size=TEXT_MAX_FONT_SIZE,
                text_max_texts=TEXT_MAX_TEXTS,
                text_min_length=TEXT_MIN_LENGTH,
                text_max_length=TEXT_MAX_LENGTH,
            )

        images.append(image)
        labels.append(mask_arr)

    inputs = processor(images=images, segmentation_maps=labels, return_tensors='pt')
    return inputs

transformed_datasets = raw_datasets.with_transform(train_transforms)


In [ ]:
#@title 10. Load model
model = SegformerForSemanticSegmentation.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable_params:,}')
print(f'Total params: {total_params:,}')


In [ ]:
#@title 11. Metrics and trainer helpers
metric = evaluate.load('mean_iou')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = torch.from_numpy(logits)

    upsampled_logits = F.interpolate(
        logits,
        size=labels.shape[-2:],
        mode='bilinear',
        align_corners=False,
    )
    pred_labels = upsampled_logits.argmax(dim=1).cpu().numpy()

    metrics = metric.compute(
        predictions=pred_labels,
        references=labels,
        num_labels=2,
        ignore_index=255,
        reduce_labels=False,
    )

    return {
        'mean_iou': metrics['mean_iou'],
        'mean_accuracy': metrics['mean_accuracy'],
        'iou_background': metrics['per_category_iou'][0],
        'iou_object': metrics['per_category_iou'][1],
    }

class ValidationLogger(TrainerCallback):
    def __init__(self, log_path='eval_results.jsonl'):
        self.log_path = log_path

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'eval_loss' in logs:
            with open(self.log_path, 'a') as f:
                payload = {'epoch': state.epoch, **logs}
                f.write(json.dumps(payload) + '\n')


In [ ]:
#@title 12. Training arguments for Colab T4
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model='mean_iou',
    greater_is_better=True,
    dataloader_num_workers=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
    bf16=False,
    weight_decay=WEIGHT_DECAY,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=transformed_datasets['train'],
    eval_dataset=transformed_datasets['test'],
    compute_metrics=compute_metrics,
    callbacks=[ValidationLogger()],
)

print(training_args)


In [ ]:
#@title 13. Start training
print('Starting training on device:', trainer.args.device)
train_result = trainer.train()
train_result


In [ ]:
#@title 14. Final evaluation
eval_metrics = trainer.evaluate()
eval_metrics


In [ ]:
#@title 15. Save best model
final_model_path = os.path.join(OUTPUT_DIR, 'final_best_model')
trainer.save_model(final_model_path)
processor.save_pretrained(final_model_path)
print('Best model saved to:', final_model_path)


In [ ]:
#@title 16. Export sample predictions
PREDICTION_DIR = Path(OUTPUT_DIR) / 'prediction_samples'
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

def predict_mask(image):
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    upsampled_logits = F.interpolate(
        outputs.logits,
        size=image.size[::-1],
        mode='bilinear',
        align_corners=False,
    )
    prediction = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
    return prediction

num_examples = min(8, len(test_records))
chosen_examples = random.sample(test_records, num_examples)

for i, example in enumerate(chosen_examples):
    image = Image.open(example['image_path']).convert('RGB')
    gt_mask = np.array(Image.open(example['mask_path']))
    pred_mask = predict_mask(image)

    image.save(PREDICTION_DIR / f'{i:02d}_image.png')
    Image.fromarray((gt_mask > 0).astype(np.uint8) * 255).save(PREDICTION_DIR / f'{i:02d}_gt.png')
    Image.fromarray(pred_mask * 255).save(PREDICTION_DIR / f'{i:02d}_pred.png')

    image_np = np.array(image).astype(np.float32)
    overlay = image_np.copy()
    red = np.array([255, 0, 0], dtype=np.float32)
    alpha = 0.7
    mask = pred_mask > 0
    overlay[mask] = (1 - alpha) * image_np[mask] + alpha * red
    overlay = overlay.astype(np.uint8)
    Image.fromarray(overlay).save(PREDICTION_DIR / f'{i:02d}_overlay.png')

print('Saved prediction samples to:', PREDICTION_DIR)


In [ ]:
#@title 17. Preview prediction samples
sample_overlays = sorted(PREDICTION_DIR.glob('*_overlay.png'))[:4]
fig, axes = plt.subplots(len(sample_overlays), 2, figsize=(12, 4 * max(1, len(sample_overlays))))
if len(sample_overlays) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, overlay_path in enumerate(sample_overlays):
    original_path = PREDICTION_DIR / overlay_path.name.replace('_overlay.png', '_image.png')

    axes[row, 0].imshow(Image.open(original_path))
    axes[row, 0].set_title(original_path.name)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(Image.open(overlay_path))
    axes[row, 1].set_title(overlay_path.name)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
#@title 18. Zip results and trigger download
# Define the zip filename
zip_filename = 'segformer_training_results.zip'

# Create a temporary directory or just zip the main output directory
# We also want to include eval_results.jsonl which is in /content/
!zip -r {zip_filename} {OUTPUT_DIR} /content/eval_results.jsonl

# Trigger the download
files.download(zip_filename)